# 01 — Data Pipeline

`demos_ant_planC.npz` (00번 산출물) 를 받아 train/val split + normalization.

**Input**:  `demos_ant_planC.npz`
**Output**: `norm_stats.npz`

**다음 노트북**: `02_vanilla_dp.ipynb` 부터 이 두 파일을 모두 사용.

## 1. Setup — Drive mount

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

import os, sys
from pathlib import Path

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import ARTIFACT_ROOT, DATA_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()

DATA_PATH = DATA_DIR / 'demos_ant_planC.npz'
NORM_PATH = DATA_DIR / 'norm_stats.npz'
assert DATA_PATH.exists(), f'파일 없음: {DATA_PATH}'
print(f'✓ src 경로 등록: {SRC_DIR}')
print(f'✓ artifact root: {ARTIFACT_ROOT}')
print(f'✓ 데이터 발견: {DATA_PATH}')

## 2. Imports + 재현성 시드

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import shutil

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__}, device={device}")

## 3. Demos 로드 + 구조 확인

In [ ]:
demos = np.load(DATA_PATH)
print(f"Keys: {list(demos.keys())}")

obs_data    = demos['observations']      # (N, T, obs_dim)
act_data    = demos['actions']           # (N, T, act_dim)
phase_data  = demos['phases']            # (N, T) — raw φ ∈ [0, 2π)
ep_lengths  = demos['episode_lengths']   # (N,)

print(f"\nobservations: {obs_data.shape}, dtype={obs_data.dtype}")
print(f"actions:      {act_data.shape}, dtype={act_data.dtype}")
print(f"phases:       {phase_data.shape}, dtype={phase_data.dtype}")
print(f"episode_lengths: min={ep_lengths.min()}, max={ep_lengths.max()}")

# 학습 frequency window (Step 0에서 저장한 metadata)
f_mean = float(demos['freq_window_mean'])
f_std  = float(demos['freq_window_std'])
f_min  = float(demos['freq_window_min'])
f_max  = float(demos['freq_window_max'])
print(f"\nLearned freq window: {f_mean:.3f} ± {f_std:.3f} Hz, "
      f"range [{f_min:.3f}, {f_max:.3f}]")
print(f"In-distribution (mean ± 2σ): "
      f"[{f_mean - 2*f_std:.3f}, {f_mean + 2*f_std:.3f}] Hz")

## 4. Hyperparameters

In [ ]:
# DP 표준 + 제안서 §5.4
OBS_HORIZON    = 2     # 과거 관측 window (현재 step 포함)
PRED_HORIZON   = 16    # action chunk 길이 H
ACTION_HORIZON = 8     # sampling 시 실행할 step 수 (re-plan 주기)

OBS_DIM = obs_data.shape[-1]   # 105
ACT_DIM = act_data.shape[-1]   # 8

VAL_RATIO  = 0.15
BATCH_SIZE = 256

print(f"OBS_DIM={OBS_DIM}, ACT_DIM={ACT_DIM}")
print(f"obs_horizon={OBS_HORIZON}, pred_horizon={PRED_HORIZON}, "
      f"action_horizon={ACTION_HORIZON}")
print(f"batch_size={BATCH_SIZE}, val_ratio={VAL_RATIO}")

## 5. Train / Val split (episode 단위)

Episode 내부 step끼리는 강한 상관관계가 있으므로 **반드시 episode 단위로 split** 해야 val leak이 없음.

In [ ]:
n_episodes = obs_data.shape[0]
n_val   = int(np.round(n_episodes * VAL_RATIO))
n_train = n_episodes - n_val

rng = np.random.RandomState(SEED)
perm = rng.permutation(n_episodes)
train_eps = perm[:n_train]
val_eps   = perm[n_train:]

print(f"Train episodes: {len(train_eps)}")
print(f"Val episodes:   {len(val_eps)}")

# Sanity: split 간 freq 분포가 비슷한지 확인 (episode별 추정 freq가 있다면)
if 'estimated_freqs' in demos:
    train_freqs = demos['estimated_freqs'][train_eps]
    val_freqs   = demos['estimated_freqs'][val_eps]
    print(f"\nTrain freq: mean={train_freqs.mean():.3f}, std={train_freqs.std():.3f}")
    print(f"Val freq:   mean={val_freqs.mean():.3f}, std={val_freqs.std():.3f}")

## 6. Normalization stats (TRAIN only)

- **Obs**: per-dim z-score `(x − μ) / σ`. 각 차원의 scale이 다르므로 (joint angle vs. velocity vs. contact force) 차원별 정규화 필수.
- **Action**: per-dim min-max → `[−1, 1]`. DDPM이 noise distribution `N(0, I)`를 가정하므로 action도 비슷한 범위로 맞춰야 학습 안정.
- **Phase**: 정규화 안 함. 모델 안에서 `(cos φ, sin φ)`로 인코딩되어 자연스럽게 `[−1, 1]`.

**중요**: Val 데이터로 stat 계산하면 정보 leak. Train만 사용.

In [ ]:
# Train 데이터만 모아서 stat 계산
train_obs_list = [obs_data[i, :int(ep_lengths[i])] for i in train_eps]
train_act_list = [act_data[i, :int(ep_lengths[i])] for i in train_eps]
train_obs_flat = np.concatenate(train_obs_list, axis=0)  # (sum_T, obs_dim)
train_act_flat = np.concatenate(train_act_list, axis=0)  # (sum_T, act_dim)

print(f"Train flat obs: {train_obs_flat.shape}")
print(f"Train flat act: {train_act_flat.shape}")

obs_mean = train_obs_flat.mean(axis=0).astype(np.float32)
obs_std  = train_obs_flat.std(axis=0).astype(np.float32)
obs_std  = np.maximum(obs_std, 1e-6)   # constant 차원 보호

act_min  = train_act_flat.min(axis=0).astype(np.float32)
act_max  = train_act_flat.max(axis=0).astype(np.float32)
# act가 saturate 안 됐을 가능성 대비해 약간 padding (선택사항이지만 안전)
act_pad  = 0.02 * (act_max - act_min)
act_min  = act_min - act_pad
act_max  = act_max + act_pad
act_range = np.maximum(act_max - act_min, 1e-6).astype(np.float32)

print(f"\nobs_mean.shape={obs_mean.shape}, "
      f"global mean of means={obs_mean.mean():.3f}")
print(f"obs_std.shape={obs_std.shape}, "
      f"global mean of stds={obs_std.mean():.3f}")
print(f"act_min={act_min}")
print(f"act_max={act_max}")

## 7. Normalization stats Drive에 저장

다음 노트북 (Vanilla DP, Phase-Conditioned DP, 평가 등) 모두 동일 stat을 써야 함. 한 번 저장 후 load만 함.

In [ ]:
norm_stats = {
    'obs_mean':  obs_mean,
    'obs_std':   obs_std,
    'act_min':   act_min,
    'act_max':   act_max,
    'act_range': act_range,
    # Frequency window metadata (평가 protocol에서 사용)
    'freq_window_mean': np.float32(f_mean),
    'freq_window_std':  np.float32(f_std),
    'freq_window_min':  np.float32(f_min),
    'freq_window_max':  np.float32(f_max),
    # Hyperparams (다음 노트북에서 일관성 검증용)
    'obs_horizon':    np.int32(OBS_HORIZON),
    'pred_horizon':   np.int32(PRED_HORIZON),
    'action_horizon': np.int32(ACTION_HORIZON),
    'obs_dim':        np.int32(OBS_DIM),
    'act_dim':        np.int32(ACT_DIM),
    # Split indices (재현성)
    'train_eps': train_eps.astype(np.int32),
    'val_eps':   val_eps.astype(np.int32),
    'seed':      np.int32(SEED),
}
np.savez(NORM_PATH, **norm_stats)
print(f"✓ 저장: {NORM_PATH}")
print(f"  파일 크기: {os.path.getsize(NORM_PATH) / 1024:.1f} KB")

## 8. `AntPhaseDataset` 정의

Chunk 단위 sample 반환:

| Key | Shape | 설명 |
|---|---|---|
| `obs`    | `(obs_horizon=2, obs_dim=105)` | 과거 2 step 관측 (정규화됨) |
| `action` | `(pred_horizon=16, act_dim=8)`  | 미래 16 step action chunk (`[−1, 1]`로 정규화됨) |
| `phase`  | `(pred_horizon=16,)`            | 미래 16 step phase chunk (raw φ, 인코딩은 모델에서) |

**Indexing 규칙**: episode 내 시점 `t`에 대해 valid 범위는 `[obs_horizon−1, episode_length−pred_horizon]`. Episode_length=200, obs_horizon=2, pred_horizon=16이면 `t ∈ [1, 184]` → episode당 184 chunks.

In [ ]:
class AntPhaseDataset(Dataset):
    """Chunked dataset for Phase-Conditioned Diffusion Policy.

    각 sample은 (obs window, action chunk, phase chunk) 튜플.
    Phase는 raw φ로 반환하고 모델에서 (cos φ, sin φ)로 인코딩.
    Vanilla DP 단계에서는 phase가 반환되지만 무시. 미리 포함시켜
    Phase-Conditioned 단계로 넘어갈 때 Dataset 코드 변경 0.
    """

    def __init__(self, obs_data, act_data, phase_data, ep_lengths,
                 episode_indices, obs_horizon, pred_horizon,
                 obs_mean, obs_std, act_min, act_range):
        self.obs_data    = obs_data
        self.act_data    = act_data
        self.phase_data  = phase_data
        self.obs_horizon = obs_horizon
        self.pred_horizon = pred_horizon

        # Stats (float32 broadcasting용)
        self.obs_mean  = obs_mean
        self.obs_std   = obs_std
        self.act_min   = act_min
        self.act_range = act_range

        # 모든 valid (ep_idx, t) 쌍 인덱싱
        self.index = []
        for ep_idx in episode_indices:
            L = int(ep_lengths[ep_idx])
            t_min = obs_horizon - 1
            t_max = L - pred_horizon  # inclusive: t + pred_horizon - 1 ≤ L − 1
            for t in range(t_min, t_max + 1):
                self.index.append((int(ep_idx), int(t)))

        print(f"  Dataset: {len(episode_indices)} episodes "
              f"→ {len(self.index)} chunks")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        ep_idx, t = self.index[idx]
        oh, ph_h = self.obs_horizon, self.pred_horizon

        obs = self.obs_data[ep_idx, t - oh + 1 : t + 1]           # (oh, obs_dim)
        act = self.act_data[ep_idx, t : t + ph_h]                  # (ph_h, act_dim)
        ph  = self.phase_data[ep_idx, t : t + ph_h]                # (ph_h,)

        # Normalize
        obs_n = (obs - self.obs_mean) / self.obs_std
        act_n = 2.0 * (act - self.act_min) / self.act_range - 1.0

        return {
            'obs':    torch.from_numpy(obs_n.astype(np.float32)),
            'action': torch.from_numpy(act_n.astype(np.float32)),
            'phase':  torch.from_numpy(ph.astype(np.float32)),
        }


print("AntPhaseDataset 정의 완료")

## 9. DataLoader 구성

In [ ]:
print("Train dataset:")
train_dataset = AntPhaseDataset(
    obs_data, act_data, phase_data, ep_lengths,
    train_eps, OBS_HORIZON, PRED_HORIZON,
    obs_mean, obs_std, act_min, act_range,
)

print("\nVal dataset:")
val_dataset = AntPhaseDataset(
    obs_data, act_data, phase_data, ep_lengths,
    val_eps, OBS_HORIZON, PRED_HORIZON,
    obs_mean, obs_std, act_min, act_range,
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True, drop_last=False,
)

print(f"\nTrain batches/epoch: {len(train_loader)}")
print(f"Val batches/epoch:   {len(val_loader)}")

## 10. Sanity check — batch 한 개 뜯어보기

확인 포인트:
- Shape이 의도대로인지
- `obs`가 거의 mean 0, std 1
- `action`이 `[−1, 1]` 안
- `phase`가 `[0, 2π)` 안

In [ ]:
batch = next(iter(train_loader))

print("=== Batch shapes & ranges ===")
for k, v in batch.items():
    print(f"  {k:8s}: shape={tuple(v.shape)}, "
          f"min={v.min():.3f}, max={v.max():.3f}, "
          f"mean={v.mean():.3f}, std={v.std():.3f}")

print("\n=== Normalization 검증 ===")
print(f"obs (목표: mean≈0, std≈1):")
print(f"  per-dim mean (앞 5개): {batch['obs'].mean(dim=(0,1))[:5].numpy().round(3)}")
print(f"  per-dim std  (앞 5개): {batch['obs'].std(dim=(0,1))[:5].numpy().round(3)}")

print(f"\naction (목표: [-1, 1] 안):")
print(f"  min={batch['action'].min():.4f}, max={batch['action'].max():.4f}")
assert batch['action'].min() >= -1.05 and batch['action'].max() <= 1.05, \
    "Action 범위 이상 — normalization 확인 필요"

print(f"\nphase (목표: [0, 2π)):")
print(f"  min={batch['phase'].min():.3f}, max={batch['phase'].max():.3f}")
assert batch['phase'].min() >= 0 and batch['phase'].max() < 2 * np.pi + 0.1, \
    "Phase 범위 이상"

print("\n✓ 모든 검증 통과")

## 11. 시각화 — sample chunk 한 개

학습 freq ≈ 2 Hz, dt = 0.05s 라 한 cycle이 약 10 step. Chunk 16 step은 약 1.6 cycle을 cover. Phase plot에 wraparound (2π → 0)이 1~2번 보이는 게 정상.

In [ ]:
sample = train_dataset[0]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Obs window (정규화된 값, 일부 차원만)
obs_show = sample['obs'].numpy()
for d in range(min(8, obs_show.shape[1])):
    axes[0].plot(obs_show[:, d], 'o-', label=f'dim {d}', markersize=4)
axes[0].set_title(f'Obs window (정규화) — shape {tuple(sample["obs"].shape)}')
axes[0].set_xlabel('Step (within window)')
axes[0].set_ylabel('Normalized value')
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(True, alpha=0.3)

# Action chunk
act_show = sample['action'].numpy()
for d in range(act_show.shape[1]):
    axes[1].plot(act_show[:, d], 'o-', label=f'a{d}', markersize=4)
axes[1].axhline(1, color='r', ls='--', alpha=0.3)
axes[1].axhline(-1, color='r', ls='--', alpha=0.3)
axes[1].set_title(f'Action chunk (정규화) — shape {tuple(sample["action"].shape)}')
axes[1].set_xlabel('Step (within chunk)')
axes[1].set_ylabel('Normalized action')
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

# Phase chunk
ph_show = sample['phase'].numpy()
axes[2].plot(ph_show, 'o-', color='purple', markersize=5)
axes[2].axhline(2 * np.pi, color='r', ls='--', alpha=0.3, label='2π')
axes[2].set_title(f'Phase chunk (raw φ) — shape {tuple(sample["phase"].shape)}')
axes[2].set_xlabel('Step (within chunk)')
axes[2].set_ylabel('φ (rad)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
out_png = FIGURES_DIR / 'data_pipeline_sanity.png'
plt.savefig(out_png, dpi=80, bbox_inches='tight')
plt.show()
print(f'✓ 저장: {out_png}')

## 12. 추가 검증 — 여러 chunk의 phase 진행률

학습 freq ≈ 2 Hz라면 **per-step phase 진행 ≈ 2π × 2 × 0.05 = 0.628 rad/step**. Chunk 안에서 phase 진행률이 일정해야 학습이 안정. 분포가 좁을수록 좋음.

In [ ]:
# 여러 chunk 무작위로 뽑아 phase 진행률 (unwrap 후 첫-끝 차이 / step수) 측정
n_check = 200
rng2 = np.random.RandomState(SEED)
sample_idxs = rng2.choice(len(train_dataset), size=n_check, replace=False)

step_advances = []
for i in sample_idxs:
    ph = train_dataset[i]['phase'].numpy()
    ph_unwrap = np.unwrap(ph)
    advance = (ph_unwrap[-1] - ph_unwrap[0]) / (len(ph) - 1)
    step_advances.append(advance)
step_advances = np.array(step_advances)

expected = 2 * np.pi * f_mean * 0.05  # rad/step at mean freq
print(f"Phase 진행률 (rad/step):")
print(f"  관측: mean={step_advances.mean():.4f}, std={step_advances.std():.4f}")
print(f"  이론: 2π × {f_mean:.3f} × 0.05 = {expected:.4f}")
print(f"  비율: {step_advances.mean()/expected:.3f}× (1.0에 가까울수록 일관)")

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.hist(step_advances, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(expected, color='r', ls='--', linewidth=2,
           label=f'Theoretical (f={f_mean:.2f}Hz): {expected:.3f}')
ax.axvline(step_advances.mean(), color='g', ls='--', linewidth=2,
           label=f'Observed mean: {step_advances.mean():.3f}')
ax.set_xlabel('Per-step phase advance (rad/step)')
ax.set_ylabel('Count')
ax.set_title(f'Phase 진행률 분포 ({n_check} chunks)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

out_png2 = FIGURES_DIR / 'data_pipeline_phase_advance.png'
plt.savefig(out_png2, dpi=80, bbox_inches='tight')
plt.show()
print(f'\n✓ 저장: {out_png2}')

## 13. 요약

| 항목 | 값 |
|---|---|
| Train episodes | 102 |
| Val episodes | 18 |
| Train chunks/epoch | ~18,700 |
| Val chunks | ~3,300 |
| Batch size | 256 |
| Steps/epoch | ~73 |

**Drive에 저장된 산출물**:
- `norm_stats.npz` — 정규화 stat + freq window metadata + split indices + hyperparams
- `data_pipeline_sanity.png` — sample chunk 시각화
- `data_pipeline_phase_advance.png` — phase 진행률 일관성 검증

**다음 노트북 (Step 2: Vanilla Diffusion Policy)에서 할 일**:
1. `demos_ant_planC.npz` + `norm_stats.npz` 로드 (이 노트북 데이터 파이프라인 그대로 재사용)
2. Conditional 1D U-Net 정의 (Chi et al. 2023 원 코드 참고)
3. DDPM training loop (cosine β schedule, 100 epoch 정도)
4. DDIM sampling (10–20 step) → unnormalize → MuJoCo Ant rollout
5. 자빠지지 않고 어느 정도 보행하는지 검증

In [ ]:
print("=== Step 1 (Data Pipeline) 완료 ===")
print(f"Train chunks: {len(train_dataset)}")
print(f"Val chunks:   {len(val_dataset)}")
print(f"Steps/epoch:  {len(train_loader)}")
print(f"\n다음 단계: Vanilla Diffusion Policy baseline (Step 2)")